# CatVTON — Local File Inference

Run virtual try-on on **local images** (no server, no ngrok, no Cloudinary).  
Just point to your person image, garment image, and optionally a mask.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 1: Clone repo (skip if already inside it)
# ═══════════════════════════════════════════════════════════════════════════════
import os
if not os.path.exists("model/pipeline.py"):
    !git clone https://github.com/usman9-ai/Virtual-Try-On.git
    os.chdir("Virtual-Try-On")
print(f"Working directory: {os.getcwd()}")

Cloning into 'Virtual-Try-On'...
remote: Enumerating objects: 879, done.
remote: Counting objects: 100% (879/879), done.
remote: Compressing objects: 100% (740/740), done.
remote: Total 879 (delta 154), reused 829 (delta 120), pack-reused 0 (from 0)
Receiving objects: 100% (879/879), 27.73 MiB | 17.83 MiB/s, done.
Resolving deltas: 100% (154/154), done.
Working directory: /content/Virtual-Try-On


In [4]:
!pip install -r /content/Virtual-Try-On/requirements.txt

  Cloning https://github.com/huggingface/diffusers.git to /tmp/pip-req-build-_i17vjuv
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /tmp/pip-req-build-_i17vjuv
  Resolved https://github.com/huggingface/diffusers.git to commit f3d42be118f9af7ed9697b686fba09a8bdcd71d1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of diffusers to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install -r /content/Virtual-Try-On/requirements.txt (line 14), -

In [1]:
!pip install fvcore av

In [2]:
!pip install peft==0.17.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 17.2 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.14.0
    Uninstalling peft-0.14.0:
      Successfully uninstalled peft-0.14.0


In [3]:
!pip install --upgrade diffusers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 25.5 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.31.0
    Uninstalling accelerate-0.31.0:
      Successfully uninstalled accelerate-0.31.0


In [12]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'Virtual-Try-On', 'sample_data']


In [18]:
!pip uninstall -y diffusers
!pip install diffusers==0.30.3

Found existing installation: diffusers 0.39.0.dev0
Uninstalling diffusers-0.39.0.dev0:
  Successfully uninstalled diffusers-0.39.0.dev0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 88.6 MB/s eta 0:00:00


In [3]:
!pip install xformers==0.0.27.post2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 808.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 7.9 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.7.0
    Uninstalling triton-3.7.0:
      Successfully uninstalled triton-3.7.0
  Attempting uninstall: torch
    Found existing installation: torch 2.12.0
    Uninstalling torch-2.12.0:
      Successfully uninstalled torch-2.12.0
  Attempting uninstall: xformers
    Found existing installation: xformers 0.0.35
    Uninstalling xformers-0.0.35:
      Successfully uninstalled xformers-0.0.35


In [1]:
%cd /content/Virtual-Try-On

/content/Virtual-Try-On


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 4: Load CatVTON Pipeline + AutoMasker
# ═══════════════════════════════════════════════════════════════════════════════

import os
import sys
sys.path.insert(0, os.getcwd())

from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor

from model.pipeline import CatVTONPipeline
from model.cloth_masker import AutoMasker
from utils import resize_and_crop, resize_and_padding, init_weight_dtype

# Download CatVTON checkpoint (contains attention adapter + DensePose + SCHP)
print("Downloading CatVTON checkpoint (first run only)...")
repo_ckpt = "zhengchong/CatVTON"
attn_folder = snapshot_download(repo_id=repo_ckpt)
print(f"Checkpoint at: {attn_folder}")

# Load pipeline
print("Loading CatVTON pipeline...")
pipeline = CatVTONPipeline(
    base_ckpt="runwayml/stable-diffusion-inpainting",
    attn_ckpt=attn_folder,
    attn_ckpt_version="mix",
    weight_dtype=init_weight_dtype("fp16"),
    use_tf32=True,
    device="cuda",
    skip_safety_check=True,
)

# Load AutoMasker (for automatic person/garment mask generation)
print("Loading AutoMasker (DensePose + SCHP)...")
mask_processor = VaeImageProcessor(
    vae_scale_factor=8,
    do_normalize=False,
    do_binarize=True,
    do_convert_grayscale=True,
)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(attn_folder, "DensePose"),
    schp_ckpt=os.path.join(attn_folder, "SCHP"),
    device="cuda",
)

print("All models loaded!")

/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

DensePose/model_final_162be9.pkl:   0%|          | 0.00/256M [00:00<?, ?B/s]

SCHP/exp-schp-201908261155-lip.pth:   0%|          | 0.00/267M [00:00<?, ?B/s]

SCHP/exp-schp-201908301523-atr.pth:   0%|          | 0.00/267M [00:00<?, ?B/s]

...

An error occurred while trying to fetch runwayml/stable-diffusion-inpainting: runwayml/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


unet/diffusion_pytorch_model.bin:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading AutoMasker (DensePose + SCHP)...


/content/Virtual-Try-On/model/SCHP/__init__.py:93: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(ckpt_path, map_location='cpu')['state_dict']


All models loaded!


In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 5: Configuration — SET YOUR FILE PATHS HERE
# ═══════════════════════════════════════════════════════════════════════════════

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ EDIT THESE PATHS to point to your local images                              │
# └─────────────────────────────────────────────────────────────────────────────┘

PERSON_IMAGE_PATH = "/content/person.jpg"        # Full-body person photo
GARMENT_IMAGE_PATH = "/content/cloth.jpeg"      # Garment image (flat-lay or on model)
MASK_IMAGE_PATH = None                           # Set to a .png path, or None for auto-mask

# Garment type: "upper" (tops/shirts), "lower" (pants/skirts), "overall" (dresses/full)
CLOTH_TYPE = "upper"

# Output settings
OUTPUT_DIR = "/content/Results"                         # Where to save results
NUM_INFERENCE_STEPS = 50                         # More steps = better quality (20-50)
GUIDANCE_SCALE = 5.0                             # Higher = more garment fidelity (2.5-7.5)
SEED = 42                                        # For reproducibility (set None for random)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Config ready. Output will be saved to: {OUTPUT_DIR}/")

Config ready. Output will be saved to: /content/Results/


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 6: Run Try-On Inference
# ═══════════════════════════════════════════════════════════════════════════════
from PIL import Image
import numpy as np
import torch

# --- Load images ---
person = Image.open(PERSON_IMAGE_PATH).convert("RGB")
cloth = Image.open(GARMENT_IMAGE_PATH).convert("RGB")
print(f"Person: {person.size}, Garment: {cloth.size}")

# --- Compute target resolution (from UNet config) ---
W = pipeline.unet.config.sample_size * 8
H = pipeline.unet.config.sample_size * 8
print(f"Processing at: {W}x{H}")

# --- Resize to model resolution ---
person = resize_and_padding(person, (W, H))
cloth = resize_and_padding(cloth, (W, H))

# --- Generate or load mask ---
if MASK_IMAGE_PATH and os.path.exists(MASK_IMAGE_PATH):
    # Use provided mask
    person_mask = Image.open(MASK_IMAGE_PATH).convert("L")
    person_mask = person_mask.resize((W, H), Image.NEAREST)
    print("Using provided mask.")
else:
    # Auto-generate mask with DensePose + SCHP
    print(f"Auto-generating person mask (cloth_type='{CLOTH_TYPE}')...")
    raw_person_mask = automasker(person, CLOTH_TYPE)["mask"]
    raw_person_mask = resize_and_padding(raw_person_mask, (W, H))
    person_mask = mask_processor.blur(raw_person_mask, blur_factor=4).convert("L")

print(f"Mask size: {person_mask.size}")

# --- Run inference ---
generator = torch.Generator(device="cuda").manual_seed(SEED) if SEED else None

print(f"Running inference ({NUM_INFERENCE_STEPS} steps, guidance={GUIDANCE_SCALE})...")
images = pipeline(
    image=person,
    condition_image=cloth,
    mask=person_mask,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
)

result_image = images[0]
print(f"Done! Result size: {result_image.size}")

Person: (3840, 5760), Garment: (853, 1280)
Processing at: 512x512
Auto-generating person mask (cloth_type='upper')...
Mask size: (512, 512)
Running inference (50 steps, guidance=5.0)...


100%|██████████| 50/50 [01:50<00:00,  2.22s/it]


Done! Result size: (768, 1024)


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 7: Display Results
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

axes[0].imshow(person)
axes[0].set_title("Person")
axes[0].axis("off")

axes[1].imshow(cloth)
axes[1].set_title("Garment")
axes[1].axis("off")

axes[2].imshow(person_mask, cmap="gray")
axes[2].set_title("Mask (auto-generated)")
axes[2].axis("off")

axes[3].imshow(result_image)
axes[3].set_title("Try-On Result")
axes[3].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 7.5: Real-ESRGAN Enhancement (cloth texture & fold sharpening)
# ═══════════════════════════════════════════════════════════════════════════════
# Self-contained Real-ESRGAN (model/enhancer.py) — only needs torch/numpy/PIL.
# `enhance_region` sharpens ONLY the garment (via person_mask) and composites
# it back, leaving the face and background untouched.
from model.enhancer import RealESRGANEnhancer

# Load once (downloads official x4 weights on first run, ~64 MB)
enhancer = RealESRGANEnhancer(
    scale=4,        # 4x model = sharpest detail (use 2 for a lighter model)
    device="cuda",
    half=True,      # fp16 — faster, less VRAM
    tile=512,       # tiled inference to bound VRAM (0 = whole image at once)
    tile_pad=32,
)
print("Real-ESRGAN enhancer loaded.")

# ── Settings ──────────────────────────────────────────────────────────────────
REGION_ONLY = True   # True = sharpen only the garment region (recommended)
OUTSCALE = 1.0       # 1.0 = same size as result; 2.0 = 2x larger output

if REGION_ONLY:
    enhanced_image = enhancer.enhance_region(
        result_image, person_mask, outscale=OUTSCALE, feather=8,
    )
    print(f"Region-only enhancement applied. Size: {enhanced_image.size}")
else:
    enhanced_image = enhancer.enhance(result_image, outscale=OUTSCALE)
    print(f"Full-image enhancement applied. Size: {enhanced_image.size}")

# Save the enhanced result alongside the raw one
import os
from pathlib import Path
_person_stem = Path(PERSON_IMAGE_PATH).stem
_garment_stem = Path(GARMENT_IMAGE_PATH).stem
_enh_path = os.path.join(OUTPUT_DIR, f"{_person_stem}_x_{_garment_stem}_enhanced.png")
enhanced_image.save(_enh_path)
print(f"Enhanced result saved to: {_enh_path}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 7.6: Compare Before / After Enhancement
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(result_image)
axes[0].set_title("Try-On Result (before)")
axes[0].axis("off")
axes[1].imshow(enhanced_image)
axes[1].set_title("Real-ESRGAN Enhanced (after)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 8: Save Result
# ═══════════════════════════════════════════════════════════════════════════════
from pathlib import Path

person_stem = Path(PERSON_IMAGE_PATH).stem
garment_stem = Path(GARMENT_IMAGE_PATH).stem
output_filename = f"{person_stem}_x_{garment_stem}_result.png"
output_path = os.path.join(OUTPUT_DIR, output_filename)

result_image.save(output_path)
print(f"Result saved to: {output_path}")

# Also save the mask for reference
mask_path = os.path.join(OUTPUT_DIR, f"{person_stem}_mask.png")
person_mask.save(mask_path)
print(f"Mask saved to:   {mask_path}")

Result saved to: /content/Results/person_x_cloth_result.png
Mask saved to:   /content/Results/person_mask.png


---
## Batch Mode: Process Multiple Garments on One Person

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 9: Batch Inference — Multiple garments on one person
# ═══════════════════════════════════════════════════════════════════════════════

def run_tryon_local(
    person_path,
    garment_path,
    mask_path=None,
    cloth_type="upper",
    steps=50,
    guidance=5.0,
    seed=42,
):
    """
    Run try-on on local files. Returns the result PIL Image.

    Args:
        person_path:  Path to person image
        garment_path: Path to garment image
        mask_path:    Path to mask image (L-mode .png), or None for auto-mask
        cloth_type:   "upper", "lower", or "overall"
        steps:        Denoising steps
        guidance:     CFG guidance scale
        seed:         Random seed (None for random)

    Returns:
        PIL.Image — the try-on result
    """
    person = Image.open(person_path).convert("RGB")
    cloth = Image.open(garment_path).convert("RGB")

    W = pipeline.unet.config.sample_size * 8
    H = pipeline.unet.config.sample_size * 8
    person = resize_and_padding(person, (W, H))
    cloth = resize_and_padding(cloth, (W, H))

    if mask_path and os.path.exists(mask_path):
        pmask = Image.open(mask_path).convert("L").resize((W, H), Image.NEAREST)
    else:
        raw = automasker(person, cloth_type)["mask"]
        raw = resize_and_padding(raw, (W, H))
        pmask = mask_processor.blur(raw, blur_factor=4).convert("L")

    gen = torch.Generator(device="cuda").manual_seed(seed) if seed else None

    images = pipeline(
        image=person,
        condition_image=cloth,
        mask=pmask,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=gen,
    )
    return images[0]


# ── Example batch usage ───────────────────────────────────────────────────────
# Uncomment and edit the paths below:

# PERSON = "data/person/model_01.jpg"
# GARMENTS = [
#     "data/garment/shirt_blue.jpg",
#     "data/garment/tshirt_white.jpg",
#     "data/garment/jacket_black.jpg",
# ]
#
# for i, gpath in enumerate(GARMENTS):
#     print(f"Processing [{i+1}/{len(GARMENTS)}]: {gpath}")
#     result = run_tryon_local(PERSON, gpath, cloth_type="upper")
#     stem = Path(gpath).stem
#     result.save(f"{OUTPUT_DIR}/batch_{stem}.png")
#     print(f"  Saved: {OUTPUT_DIR}/batch_{stem}.png")
#
# print("Batch complete!")

print("Batch helper ready — uncomment the example above and set your paths.")

---
## Process an Entire Folder

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 10: Folder-based batch — person/ + garment/ directories
# ═══════════════════════════════════════════════════════════════════════════════

def run_tryon_folder(
    person_dir,
    garment_dir,
    mask_dir=None,
    output_dir="./results",
    cloth_type="upper",
    steps=50,
    guidance=5.0,
    seed=42,
):
    """
    Process all matching person-garment pairs from folders.

    Expects same filenames in person_dir/ and garment_dir/.
    Optionally reads masks from mask_dir/ (same filename, .png).
    """
    os.makedirs(output_dir, exist_ok=True)

    person_files = sorted([
        f for f in os.listdir(person_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    print(f"Found {len(person_files)} person images in {person_dir}")

    for i, fname in enumerate(person_files):
        person_path = os.path.join(person_dir, fname)
        garment_path = os.path.join(garment_dir, fname)

        if not os.path.exists(garment_path):
            # Try with different extension
            stem = Path(fname).stem
            garment_path = None
            for ext in [".jpg", ".jpeg", ".png"]:
                candidate = os.path.join(garment_dir, stem + ext)
                if os.path.exists(candidate):
                    garment_path = candidate
                    break
            if garment_path is None:
                print(f"  [{i+1}] SKIP (no matching garment): {fname}")
                continue

        mask_path = None
        if mask_dir:
            stem = Path(fname).stem
            mp = os.path.join(mask_dir, stem + ".png")
            if os.path.exists(mp):
                mask_path = mp

        print(f"  [{i+1}/{len(person_files)}] {fname} ...", end=" ")
        result = run_tryon_local(
            person_path, garment_path, mask_path,
            cloth_type=cloth_type, steps=steps, guidance=guidance, seed=seed,
        )

        out_path = os.path.join(output_dir, Path(fname).stem + "_result.png")
        result.save(out_path)
        print(f"saved.")

    print(f"\nAll done! Results in: {output_dir}/")


# ── Example usage ─────────────────────────────────────────────────────────────
# Uncomment and edit:

# run_tryon_folder(
#     person_dir="./data/person",
#     garment_dir="./data/garment",
#     mask_dir="./data/mask",       # or None for auto-mask
#     output_dir="./results",
#     cloth_type="upper",
#     steps=50,
#     guidance=5.0,
#     seed=42,
# )

print("Folder processor ready — uncomment above to run on a dataset.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 11: VRAM usage info
# ═══════════════════════════════════════════════════════════════════════════════
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"VRAM total:     {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

---
# FLUX Try-On (higher-quality path)

The CatVTON cells above stay as-is. The cells below run the **FLUX.1-Fill-dev**
try-on path through the worker's own loader (`worker/inference_engine.py`), with
a **fine-tuned try-on transformer** so the garment is actually used.

**Why your earlier Flux output was grey:** with no try-on transformer set, the
pipeline loads the *base* FLUX.1-Fill-dev, which is a generic inpainter that
ignores the garment reference — so the AutoMasker hole gets filled with a flat
grey. Loading `xiaozaa/catvton-flux-alpha` (Option A) fixes this.

> **Run order:** Restart the runtime before running this section so the CatVTON
> model isn't also holding VRAM (or run the "free VRAM" cell below). You need a
> Hugging Face token with access to the gated `black-forest-labs/FLUX.1-Fill-dev`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Flux Cell 1: GPU / VRAM probe — decide Option A (full transformer) vs B (LoRA)
# ═══════════════════════════════════════════════════════════════════════════════
import torch

if not torch.cuda.is_available():
    print("No CUDA GPU visible — switch the runtime to a GPU (Colab: T4).")
else:
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    props = torch.cuda.get_device_properties(0)
    total = props.total_memory / 1024**3
    free_b, _ = torch.cuda.mem_get_info()
    free = free_b / 1024**3
    print(f"GPU:           {name}")
    print(f"Compute cap:   sm_{major}{minor}  (native bf16: {'yes' if major >= 8 else 'no -> uses fp16'})")
    print(f"Total VRAM:    {total:.1f} GB")
    print(f"Free VRAM now: {free:.1f} GB")
    print("-" * 60)
    print("RECOMMENDATION:")
    if total >= 22:
        print("  Option A (full transformer 'xiaozaa/catvton-flux-alpha').")
        print("  You can even disable 4-bit quant (--no-quant) for max quality.")
    elif total >= 13:
        print("  Option A (full transformer) WITH NF4 4-bit quantization.")
        print("  Fits a 16GB T4. Best quality for the VRAM. -> use the cells below as-is.")
    else:
        print("  Option B (try-on LoRA on base Fill). The full transformer may OOM here.")
        print("  In the load cell set FLUX_TRANSFORMER_CKPT='' and FLUX_LORA_PATH=<lora repo>.")
    if free < 6 and total >= 13:
        print("\n  NOTE: free VRAM is low — run the 'free VRAM' cell or restart the runtime first.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Flux Cell 2: Setup — HF login, bitsandbytes, and free CatVTON VRAM (optional)
# ═══════════════════════════════════════════════════════════════════════════════
# 1) bitsandbytes is required for NF4 4-bit quantization (Option A on a T4).
!pip install -q bitsandbytes

# 2) Log in to Hugging Face (needs access to gated black-forest-labs/FLUX.1-Fill-dev).
from huggingface_hub import login
login()  # paste your token when prompted

# 3) Free the CatVTON pipeline if it is still loaded, to reclaim VRAM for Flux.
import gc, torch
if "pipeline" in globals():
    try:
        del pipeline
        print("Freed CatVTON `pipeline` from memory.")
    except Exception as e:
        print("Could not free CatVTON pipeline:", e)
gc.collect()
torch.cuda.empty_cache()
print("Setup done.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Flux Cell 3: Load AutoMasker + build the FLUX try-on pipeline (Option A)
# ═══════════════════════════════════════════════════════════════════════════════
import os, torch
from huggingface_hub import snapshot_download

from worker.inference_engine import InferenceEngine
from worker.models import InferenceConfig
from model.flux.pipeline_flux_tryon import FluxTryOnPipeline

# --- Flux target resolution (must be multiples of 16) ---
FLUX_W = 768
FLUX_H = 1024

# --- Option A: fine-tuned try-on transformer (fixes the grey output) ---------
# This repo ships the transformer at the ROOT (subfolder = "").
FLUX_TRANSFORMER = "xiaozaa/catvton-flux-alpha"
FLUX_TRANSFORMER_SUBFOLDER = ""
FLUX_LORA = ""                      # leave empty for Option A
# For Option B instead: set FLUX_TRANSFORMER = "" and FLUX_LORA = "<lora-repo-or-path>".

flux_config = InferenceConfig(
    pipeline_type="flux",
    device="cuda:0",
    batch_size=1,
    flash_attention=False,
    vae_tiling=True,
    vae_slicing=False,
    vae_tiling_resolution=1024,
    flux_base_ckpt="black-forest-labs/FLUX.1-Fill-dev",
    flux_transformer_ckpt=FLUX_TRANSFORMER,
    flux_transformer_subfolder=FLUX_TRANSFORMER_SUBFOLDER,
    flux_lora_path=FLUX_LORA,
    flux_quantize_4bit=True,        # NF4 -> fits a 16GB T4
    flux_vae_fp32=True,             # fp32 VAE -> avoids black/NaN decodes on T4
    flux_cpu_offload="none",
    flux_height=FLUX_H,
    flux_width=FLUX_W,
)

# Minimal shim so we can reuse the worker loader without the worker env/config.
class _Cfg:
    FLUX_BASE_CKPT = flux_config.flux_base_ckpt
    FLUX_CKPT = ""

_engine = object.__new__(InferenceEngine)
_engine.config = flux_config

print("Building Flux pipeline (first run downloads weights — a few minutes)...")
flux_pipeline = _engine._build_flux_pipeline(flux_config, _Cfg, FluxTryOnPipeline, torch)
if flux_config.vae_tiling:
    flux_pipeline.enable_vae_tiling()
print("Flux pipeline ready.")

# --- AutoMasker (reuse if already loaded by the CatVTON cells) ---------------
try:
    automasker
    print("Reusing AutoMasker already in memory.")
except NameError:
    from model.cloth_masker import AutoMasker
    print("Loading AutoMasker (DensePose + SCHP)...")
    _ckpt = snapshot_download(repo_id="zhengchong/CatVTON")
    automasker = AutoMasker(
        densepose_ckpt=os.path.join(_ckpt, "DensePose"),
        schp_ckpt=os.path.join(_ckpt, "SCHP"),
        device="cuda",
    )
    print("AutoMasker ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Flux Cell 4: Configuration — set your inputs here
# ═══════════════════════════════════════════════════════════════════════════════
FLUX_PERSON_PATH  = "/content/person.jpg"     # full-body person photo
FLUX_GARMENT_PATH = "/content/cloth.jpeg"     # garment image
FLUX_MASK_PATH    = None                       # set a .png path to override AutoMasker, else None
FLUX_CLOTH_TYPE   = "upper"                    # upper | lower | overall | inner | outer

# Sampling
FLUX_STEPS        = 30
FLUX_GUIDANCE     = 30.0
FLUX_SEED         = 42                          # set None for random

# Mask post-processing / composite
FLUX_DILATION_PX        = 0                     # expand mask outward (px)
FLUX_FEATHER_PX         = 0                     # soften mask edge fed to the model (px)
FLUX_COMPOSITE          = True                  # paste original face/pants/bg back (pixel-identical outside mask)
FLUX_COMPOSITE_FEATHER  = 6                     # blend band for the composite seam (px, 0 = hard)

FLUX_OUTPUT_DIR = "/content/Results"
os.makedirs(FLUX_OUTPUT_DIR, exist_ok=True)
print("Flux config set.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Flux Cell 5: Run FLUX try-on (mask -> inference -> composite -> display/save)
# ═══════════════════════════════════════════════════════════════════════════════
import torch, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from worker.mask_utils import align_to_multiple_of_16, composite_with_mask, dilate_mask, feather_mask

# --- Load + resize inputs to aligned Flux dimensions ---
person = Image.open(FLUX_PERSON_PATH).convert("RGB")
cloth  = Image.open(FLUX_GARMENT_PATH).convert("RGB")
flux_w, flux_h = align_to_multiple_of_16(FLUX_W, FLUX_H)
person_r = person.resize((flux_w, flux_h), Image.LANCZOS)
cloth_r  = cloth.resize((flux_w, flux_h), Image.LANCZOS)

# --- Mask: manual override, else AutoMasker (cloth_type-aware) ---
if FLUX_MASK_PATH and os.path.exists(FLUX_MASK_PATH):
    mask = Image.open(FLUX_MASK_PATH).convert("L").resize((flux_w, flux_h), Image.NEAREST)
    print("Using manual mask.")
else:
    print(f"Generating AutoMasker mask (cloth_type='{FLUX_CLOTH_TYPE}')...")
    mask = automasker(person_r, FLUX_CLOTH_TYPE)["mask"].convert("L").resize((flux_w, flux_h), Image.NEAREST)
    if FLUX_DILATION_PX > 0:
        mask = dilate_mask(mask, FLUX_DILATION_PX)
    if FLUX_FEATHER_PX > 0:
        mask = feather_mask(mask, FLUX_FEATHER_PX)

# --- Inference ---
gen = torch.Generator("cuda").manual_seed(FLUX_SEED) if FLUX_SEED is not None else None
print(f"Running Flux: {FLUX_STEPS} steps, guidance={FLUX_GUIDANCE}, {flux_w}x{flux_h} ...")
out = flux_pipeline(
    image=person_r,
    condition_image=cloth_r,
    mask_image=mask,
    height=flux_h,
    width=flux_w,
    num_inference_steps=FLUX_STEPS,
    guidance_scale=FLUX_GUIDANCE,
    generator=gen,
)
flux_raw = out.images[0] if hasattr(out, "images") else out[0]

# --- Composite: keep face/pants/background pixel-identical to the original ---
if FLUX_COMPOSITE:
    flux_result = composite_with_mask(person_r, flux_raw, mask, feather_px=FLUX_COMPOSITE_FEATHER)
else:
    flux_result = flux_raw

# --- Grey/black sanity check ---
a = np.asarray(flux_raw.convert("RGB")).astype(np.float32)
print(f"Output stats: mean={a.mean():.1f}, std={a.std():.1f}")
if a.std() < 12:
    print("WARNING: very low variance -> likely still a flat/grey fill. "
          "Confirm the try-on transformer loaded (Flux Cell 3) and HF login succeeded.")

# --- Display ---
fig, ax = plt.subplots(1, 4, figsize=(18, 7))
for a_, im, t in zip(ax, [person_r, cloth_r, mask, flux_result],
                     ["Person", "Garment", "Mask", "Flux Try-On"]):
    a_.imshow(im, cmap="gray" if im.mode == "L" else None)
    a_.set_title(t); a_.axis("off")
plt.tight_layout(); plt.show()

# --- Save ---
_stem = f"{Path(FLUX_PERSON_PATH).stem}_x_{Path(FLUX_GARMENT_PATH).stem}_flux.png"
_out = os.path.join(FLUX_OUTPUT_DIR, _stem)
flux_result.save(_out)
print(f"Saved: {_out}")

---
### Flux Diagnostic: rectangle vs tight-garment vs agnostic mask

Runs the **same** loaded Flux pipeline on the **same person + garment** with three
different masks, to localize the grey issue:

- **rectangle** — the center-box mask the smoke test used when Flux "worked before".
- **tight** — only the segmented garment region (lightly dilated), not the whole torso+arms.
- **agnostic** — the full AutoMasker mask (the one that came out grey).

Read the printed `mean/std` per result: a proper garment has **std > ~20**; a flat grey
fill has **std < ~10**. This runs Flux up to 3 times, so it takes a few minutes on a T4.

> Requires Flux Cell 3 (pipeline + automasker) and Flux Cell 4 (config) to have run.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Flux Diagnostic: compare rectangle / tight / agnostic masks through one pipeline
# ═══════════════════════════════════════════════════════════════════════════════
import os, numpy as np, cv2, torch
import matplotlib.pyplot as plt
from PIL import Image
from worker.mask_utils import align_to_multiple_of_16, composite_with_mask
from model.cloth_masker import part_mask_of, ATR_MAPPING, LIP_MAPPING, MASK_CLOTH_PARTS

# Toggle which masks to actually run through Flux (each adds a pass).
RUN_RECTANGLE = True
RUN_TIGHT     = True
RUN_AGNOSTIC  = False   # set True to also re-run the full agnostic mask (you already saw it grey)

# Where to save everything (falls back to /content/Results if FLUX_OUTPUT_DIR is unset).
DIAG_DIR = os.path.join(globals().get("FLUX_OUTPUT_DIR", "/content/Results"), "diagnostic")
os.makedirs(DIAG_DIR, exist_ok=True)

flux_w, flux_h = align_to_multiple_of_16(FLUX_W, FLUX_H)
person = Image.open(FLUX_PERSON_PATH).convert("RGB")
cloth  = Image.open(FLUX_GARMENT_PATH).convert("RGB")
person_r = person.resize((flux_w, flux_h), Image.LANCZOS)
cloth_r  = cloth.resize((flux_w, flux_h), Image.LANCZOS)

# --- Build the three candidate masks (all at flux_w x flux_h, white = inpaint) ---
# 1) rectangle (the "worked before" control)
rect = np.zeros((flux_h, flux_w), np.uint8)
rect[int(flux_h * 0.20):int(flux_h * 0.70), int(flux_w * 0.25):int(flux_w * 0.75)] = 255
mask_rect = Image.fromarray(rect, mode="L")

# 2) + 3) from AutoMasker preprocessing (one DensePose/SCHP pass)
amres = automasker(person_r, FLUX_CLOTH_TYPE)
mask_agnostic = amres["mask"].convert("L").resize((flux_w, flux_h), Image.NEAREST)

atr = np.array(amres["schp_atr"]); lip = np.array(amres["schp_lip"])
parts = MASK_CLOTH_PARTS[FLUX_CLOTH_TYPE]
tight = (part_mask_of(parts, atr, ATR_MAPPING) | part_mask_of(parts, lip, LIP_MAPPING)).astype(np.uint8) * 255
tight = cv2.dilate(tight, np.ones((9, 9), np.uint8), iterations=2)   # small margin around the garment
mask_tight = Image.fromarray(tight).convert("L").resize((flux_w, flux_h), Image.NEAREST)

# --- Save the inputs + all three masks up-front so you can inspect them ---
person_r.save(os.path.join(DIAG_DIR, "input_person.png"))
cloth_r.save(os.path.join(DIAG_DIR, "input_garment.png"))
mask_rect.save(os.path.join(DIAG_DIR, "mask_rectangle.png"))
mask_tight.save(os.path.join(DIAG_DIR, "mask_tight.png"))
mask_agnostic.save(os.path.join(DIAG_DIR, "mask_agnostic.png"))
print(f"Saved inputs + masks to: {DIAG_DIR}")


def _run(mask):
    gen = torch.Generator("cuda").manual_seed(FLUX_SEED) if FLUX_SEED is not None else None
    out = flux_pipeline(image=person_r, condition_image=cloth_r, mask_image=mask,
                        height=flux_h, width=flux_w,
                        num_inference_steps=FLUX_STEPS, guidance_scale=FLUX_GUIDANCE,
                        generator=gen)
    raw = out.images[0] if hasattr(out, "images") else out[0]
    comp = composite_with_mask(person_r, raw, mask, feather_px=FLUX_COMPOSITE_FEATHER) if FLUX_COMPOSITE else raw
    a = np.asarray(raw.convert("RGB")).astype(np.float32)
    return raw, comp, float(a.mean()), float(a.std())


runs = []
if RUN_RECTANGLE: runs.append(("rectangle", mask_rect))
if RUN_TIGHT:     runs.append(("tight",     mask_tight))
if RUN_AGNOSTIC:  runs.append(("agnostic",  mask_agnostic))

results = []
for name, m in runs:
    print(f"Running Flux with '{name}' mask ...")
    raw, img, mean, std = _run(m)
    verdict = "FLAT GREY (garment not generated)" if std < 12 else "garment generated"
    print(f"  {name}: mean={mean:.1f}, std={std:.1f}  ->  {verdict}")
    # Save both the raw model output and the composited result for this mask.
    raw.save(os.path.join(DIAG_DIR, f"result_{name}_raw.png"))
    img.save(os.path.join(DIAG_DIR, f"result_{name}_composite.png"))
    print(f"  saved: result_{name}_raw.png, result_{name}_composite.png")
    results.append((name, img))

# --- Display masks + results, and SAVE the combined comparison figure ---
n = len(results)
fig, ax = plt.subplots(2, max(3, n), figsize=(5 * max(3, n), 12))
for a_, im, t in zip(ax[0], [mask_rect, mask_tight, mask_agnostic],
                     ["rectangle mask", "tight mask", "agnostic mask"]):
    a_.imshow(im, cmap="gray"); a_.set_title(t); a_.axis("off")
for j in range(n):
    ax[1][j].imshow(results[j][1]); ax[1][j].set_title(f"{results[j][0]} result"); ax[1][j].axis("off")
for j in range(n, max(3, n)):
    ax[1][j].axis("off")
plt.tight_layout()
_fig_path = os.path.join(DIAG_DIR, "comparison.png")
fig.savefig(_fig_path, dpi=110, bbox_inches="tight")
plt.show()
print(f"\nSaved comparison figure to: {_fig_path}")

print("\nINTERPRETATION:")
print("  rectangle good + agnostic grey -> pipeline OK; agnostic hole too big. Use the 'tight' mask.")
print("  tight good                     -> switch your run cell to the tight mask (fix below).")
print("  all grey                       -> pipeline/transformer load issue, not the mask.")

---
# FLUX Try-On — OFFICIAL recipe, T4-fit (cached prompt embeddings)

Real try-on with `catvton-flux-alpha` + AutoMasker on a **free T4**, by never
holding the T5 text encoder and the transformer in memory at the same time:

1. **Cell B** loads only the text encoders, encodes the fixed try-on prompt **once**,
   saves the embeddings to disk, then frees the encoders.
2. **Cell C** loads only the NF4 transformer + VAE (no T5) — fits the T4.
3. **Cell D** runs FluxFill using the cached embeddings (garment-left / person-right
   layout, right half = result).

Run on a fresh runtime, in order A -> B -> C -> D. Needs HF access to
`black-forest-labs/FLUX.1-Fill-dev`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Official Cell A: dependencies + HF login  (RESTART runtime after this, then run B)
# ═══════════════════════════════════════════════════════════════════════════════
# FluxFillPipeline needs diffusers>=0.32; 8-bit T5 needs transformers>=4.47
# (older transformers caused the "is_fsdp_enabled" import error you saw).
!pip install -q -U "diffusers>=0.32.2" "transformers>=4.47" "accelerate>=1.0" bitsandbytes sentencepiece protobuf

from huggingface_hub import login
login()  # paste your token (needs FLUX.1-Fill-dev access)

import gc, torch
for _n in ("pipeline", "flux_pipeline", "pipe"):
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
print("Deps installed + logged in. If diffusers/transformers were UPGRADED above, do Runtime > Restart, then run Cell B.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Official Cell B: config + encode the fixed prompt ONCE (text encoders only)
# ═══════════════════════════════════════════════════════════════════════════════
import os, gc, torch
from transformers import CLIPTextModel, CLIPTokenizer, T5EncoderModel, T5TokenizerFast
from transformers import BitsAndBytesConfig as TransformersBnb

# ---- models ----
BASE = "black-forest-labs/FLUX.1-Fill-dev"        # VAE/scheduler/text-encoders/tokenizers
TRYON_TRANSFORMER = "xiaozaa/catvton-flux-alpha"  # fine-tuned try-on transformer (repo root)
DT = torch.float16                                 # T4 has no bf16

# ---- inputs / run settings (edit these) ----
OFF_PERSON  = globals().get("FLUX_PERSON_PATH",  "/content/person.jpg")
OFF_GARMENT = globals().get("FLUX_GARMENT_PATH", "/content/cloth.jpeg")
OFF_MASK_PATH = None                  # .png to override AutoMasker, else None
OFF_CLOTH_TYPE = "upper"              # upper | lower | overall | inner | outer
OFF_W, OFF_H = 576, 768               # per-side size; canvas runs at OFF_W*2 wide
OFF_STEPS = 30
OFF_GUIDANCE = 30.0
OFF_SEED = 42                         # None for random
OFF_COMPOSITE = True
OFF_COMPOSITE_FEATHER = 6
OFF_OUTPUT_DIR = os.path.join(globals().get("FLUX_OUTPUT_DIR", "/content/Results"), "flux_official")
os.makedirs(OFF_OUTPUT_DIR, exist_ok=True)
EMB_PATH = "/content/flux_prompt_embeds.pt"

# The exact prompt catvton-flux-alpha was trained with (do not change).
PROMPT = ("The pair of images highlights a clothing and its styling on a model, "
          "high resolution, 4K, 8K; [IMAGE1] Detailed product shot of a clothing"
          "[IMAGE2] The same cloth is worn by a model in a lifestyle setting.")

# ---- encode the prompt with the text encoders only (no transformer in memory) ----
print("Loading CLIP (fp16) + T5 (8-bit) to encode the fixed prompt once ...")
tok  = CLIPTokenizer.from_pretrained(BASE, subfolder="tokenizer")
tok2 = T5TokenizerFast.from_pretrained(BASE, subfolder="tokenizer_2")
te   = CLIPTextModel.from_pretrained(BASE, subfolder="text_encoder", torch_dtype=DT).to("cuda")
te2  = T5EncoderModel.from_pretrained(
    BASE, subfolder="text_encoder_2",
    quantization_config=TransformersBnb(load_in_8bit=True), torch_dtype=DT, device_map={"": 0},
)

with torch.no_grad():
    clip_ids = tok([PROMPT], padding="max_length", max_length=77, truncation=True,
                   return_tensors="pt").input_ids.to("cuda")
    pooled = te(clip_ids).pooler_output                                   # [1, 768]
    t5_ids = tok2([PROMPT], padding="max_length", max_length=512, truncation=True,
                  return_tensors="pt").input_ids.to("cuda")
    prompt_embeds = te2(t5_ids)[0]                                        # [1, 512, 4096]

torch.save({"prompt_embeds": prompt_embeds.to(DT).cpu(),
            "pooled_prompt_embeds": pooled.to(DT).cpu()}, EMB_PATH)
print(f"Saved embeddings to {EMB_PATH} | prompt_embeds={tuple(prompt_embeds.shape)}, pooled={tuple(pooled.shape)}")

# ---- free the text encoders so the transformer can load without RAM pressure ----
del te, te2, tok, tok2, clip_ids, t5_ids, prompt_embeds, pooled
gc.collect(); torch.cuda.empty_cache()
print("Text encoders freed. Now run Cell C.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Official Cell C: load ONLY the NF4 transformer + VAE (no T5) + AutoMasker
# ═══════════════════════════════════════════════════════════════════════════════
import os, torch
from huggingface_hub import snapshot_download
from diffusers import FluxFillPipeline, FluxTransformer2DModel, AutoencoderKL
from diffusers import BitsAndBytesConfig as DiffusersBnb

print("Loading try-on transformer in NF4 4-bit ...")
transformer = FluxTransformer2DModel.from_pretrained(
    TRYON_TRANSFORMER,
    quantization_config=DiffusersBnb(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                     bnb_4bit_compute_dtype=DT),
    torch_dtype=DT,
)
print("Loading VAE in fp32 (T4 decode safety) ...")
vae = AutoencoderKL.from_pretrained(BASE, subfolder="vae", torch_dtype=torch.float32)

print("Assembling FluxFillPipeline WITHOUT text encoders (cached embeds will be used) ...")
pipe = FluxFillPipeline.from_pretrained(
    BASE,
    transformer=transformer,
    vae=vae,
    text_encoder=None, tokenizer=None,
    text_encoder_2=None, tokenizer_2=None,
    torch_dtype=DT,
)
try:
    pipe.to("cuda")
except Exception as e:
    print("pipe.to('cuda') note (quantized component stays put):", e)
    pipe.vae.to("cuda")
pipe.enable_vae_tiling()      # bound VRAM for the wide garment|person canvas
print("Transformer-only FluxFill pipeline ready — no T5 in memory.")

# AutoMasker (reuse if loaded, else fetch DensePose+SCHP from the CatVTON repo)
try:
    automasker
    print("Reusing AutoMasker.")
except NameError:
    from model.cloth_masker import AutoMasker
    print("Loading AutoMasker (DensePose + SCHP)...")
    _ckpt = snapshot_download(repo_id="zhengchong/CatVTON")
    automasker = AutoMasker(densepose_ckpt=os.path.join(_ckpt, "DensePose"),
                            schp_ckpt=os.path.join(_ckpt, "SCHP"), device="cuda")
    print("AutoMasker ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Official Cell D: run try-on with the cached prompt embeddings
# ═══════════════════════════════════════════════════════════════════════════════
import os, torch, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from torchvision import transforms
from worker.mask_utils import composite_with_mask

size = (OFF_W, OFF_H)  # (width, height) per panel
person  = Image.open(OFF_PERSON).convert("RGB").resize(size)
garment = Image.open(OFF_GARMENT).convert("RGB").resize(size)

# Person-side mask (white = inpaint)
if OFF_MASK_PATH and os.path.exists(OFF_MASK_PATH):
    mask_img = Image.open(OFF_MASK_PATH).convert("L").resize(size, Image.NEAREST)
    print("Using manual mask.")
else:
    print(f"Generating AutoMasker mask (cloth_type='{OFF_CLOTH_TYPE}')...")
    mask_img = automasker(person, OFF_CLOTH_TYPE)["mask"].convert("L").resize(size, Image.NEAREST)

# Official layout: [garment | person] along width; mask only on the person panel.
to_t = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
to_m = transforms.ToTensor()
image_t   = to_t(person)
garment_t = to_t(garment)
mask_t    = to_m(mask_img)[:1]
inpaint_image = torch.cat([garment_t, image_t], dim=2)                  # [3,H,2W]
extended_mask = torch.cat([torch.zeros_like(mask_t), mask_t], dim=2)    # [1,H,2W]

# Cached prompt embeddings (no T5 needed at inference)
emb = torch.load(EMB_PATH)
pe  = emb["prompt_embeds"].to("cuda", DT)
ppe = emb["pooled_prompt_embeds"].to("cuda", DT)

gen = torch.Generator("cpu").manual_seed(OFF_SEED) if OFF_SEED is not None else None
print(f"Running FluxFill (cached prompt): {OFF_STEPS} steps, guidance={OFF_GUIDANCE}, "
      f"canvas={size[0]*2}x{size[1]} (slow on T4)...")
result = pipe(
    image=inpaint_image,
    mask_image=extended_mask,
    height=size[1],
    width=size[0] * 2,
    prompt_embeds=pe,
    pooled_prompt_embeds=ppe,
    num_inference_steps=OFF_STEPS,
    guidance_scale=OFF_GUIDANCE,
    max_sequence_length=512,
    generator=gen,
).images[0]

# Try-on is the RIGHT panel
w = size[0]
tryon = result.crop((w, 0, w * 2, size[1]))
final = composite_with_mask(person, tryon, mask_img, feather_px=OFF_COMPOSITE_FEATHER) if OFF_COMPOSITE else tryon

a = np.asarray(tryon.convert("RGB")).astype(np.float32)
print(f"Try-on stats: mean={a.mean():.1f}, std={a.std():.1f}  ->  "
      f"{'FLAT GREY (check prompt embeds loaded)' if a.std() < 12 else 'garment generated'}")

stem = f"{Path(OFF_PERSON).stem}_x_{Path(OFF_GARMENT).stem}"
mask_img.save(os.path.join(OFF_OUTPUT_DIR, f"{stem}_mask.png"))
tryon.save(os.path.join(OFF_OUTPUT_DIR, f"{stem}_tryon_raw.png"))
final.save(os.path.join(OFF_OUTPUT_DIR, f"{stem}_tryon_final.png"))
print("Saved to:", OFF_OUTPUT_DIR)

fig, ax = plt.subplots(1, 4, figsize=(18, 7))
for a_, im, t in zip(ax, [person, garment, mask_img, final],
                     ["Person", "Garment", "Mask", "Try-On (official)"]):
    a_.imshow(im, cmap="gray" if im.mode == "L" else None); a_.set_title(t); a_.axis("off")
plt.tight_layout(); plt.show()